### Simple Agents with Ollama Model

[Documentation Link](https://strandsagents.com/docs/user-guide/concepts/model-providers/ollama/)

#### Importing required libraries

In [10]:
from strands import Agent
from strands.models.ollama import OllamaModel
from strands.multiagent.base import AgentResult, MultiAgentBase, MultiAgentResult, NodeResult
from strands.multiagent.graph import GraphResult, GraphBuilder, Status
from strands.types.content import ContentBlock, Message
from pydantic import BaseModel, Field
from typing import Optional, Any

#### Configure local Ollama model

In [3]:
# let's configure local ollama model - not all parameters are required - adjust as per need
model = OllamaModel(
    host="http://localhost:11434",  # ollama runs on this / The address of the Ollama server
    model_id="llama3.2:1b",         # The Ollama model identifier
    temperature=0.4,                # Controls randomness (higher = more random)
    max_tokens=2048,                # Maximum number of tokens to generate
    keep_alive="10m",               # How long the model stays loaded in memory
    top_p=0.8,                      # Controls diversity via nucleus sampling
    stop_sequences=["###", "END"],  # List of sequences that stop generation
    options={"top_k": 40}           # Additional model parameters (e.g., top_k)
)

#### creating nodes for graph

In [7]:
class FunctionNode(MultiAgentBase):
    def __init__(self, func):
        super().__init__()
        self.func = func
        self.name = func.__name__

    async def invoke_async(self, task, invocation_state = None, **kwargs):
        output = self.func(task, invocation_state)

        agent_result = AgentResult(
            stop_reason="end_turn",
            message=Message(role="assistant", content=[ContentBlock(text=str(output))]),
            metrics=None
        )

        return MultiAgentResult(
            status=Status.COMPLETED,
            results={self.name: NodeResult(result=agent_result)}
            # ... execution details
        )

In [11]:
class QueryAnalyzerOutput(BaseModel):
    needs_db_call: bool = Field(..., description="A falg to determine if db call is required or not")
    needs_clarification: bool = Field(..., description="A falg to determine if some clarification required to understand the user query")
    clarification_question: Optional[str] = Field(default="", description="Required if 'needs_clarification' flag is True. A related question to better understand the user ask")
    answer_from_history: Optional[str] = Field(default="", description="A well formatted answer IF AND ONLY IF can be determined from previous chat history.") 

In [ ]:

def query_analyzer_node(task, invocation_state):

    agent = Agent(model=model,
                  system_prompt="Analyze query and identify if db call is required",
                  structured_output_model=QueryAnalyzerOutput)

    _response = agent(invocation_state["query"])

    _output: QueryAnalyzerOutput = _response.structured_output



In [ ]:

def call_database_node(task, invocation_state): 

    invocation_state["database_records"] = []
    return {"call_database_node": Status.COMPLETED}

    

In [ ]:
response  = agent("What is difference between Assistive AI and Agentic AI?")
print(response)

Assistive AI and agentic AI are two distinct categories of artificial intelligence (AI) that differ in their approach, goals, and characteristics.

**Agentic AI**

Agentic AI refers to a type of AI that aims to augment human agency and autonomy. It seeks to enable humans to make decisions, take actions, or perform tasks autonomously, often with the goal of improving productivity, efficiency, or decision-making processes. Agentic AI can be seen as a form of "smart assistance" where machines provide support and guidance to humans, but ultimately allow them to operate independently.

Examples of agentic AI include:

* Virtual assistants like Siri, Alexa, or Google Assistant
* Self-driving cars that enable human drivers to focus on other tasks while the vehicle operates autonomously
* Personalized recommendation systems that suggest products or services based on individual preferences

**Assistive AI**

Assistive AI, on the other hand, focuses on enhancing human capabilities and abilities 

In [ ]:
agent.messages

[{'role': 'user',
  'content': [{'text': 'What is difference between Assistive AI and Agentic AI?'}],
  'tracking_id': 'b4d59965-00ba-4877-8d88-7b0d62d45378'},
 {'role': 'assistant',
  'content': [{'text': 'Assistive AI and agentic AI are two distinct categories of artificial intelligence (AI) that differ in their approach, goals, and characteristics.\n\n**Agentic AI**\n\nAgentic AI refers to a type of AI that aims to augment human agency and autonomy. It seeks to enable humans to make decisions, take actions, or perform tasks autonomously, often with the goal of improving productivity, efficiency, or decision-making processes. Agentic AI can be seen as a form of "smart assistance" where machines provide support and guidance to humans, but ultimately allow them to operate independently.\n\nExamples of agentic AI include:\n\n* Virtual assistants like Siri, Alexa, or Google Assistant\n* Self-driving cars that enable human drivers to focus on other tasks while the vehicle operates autonom